In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, Dropdown, IntSlider, FloatSlider, HBox, VBox, HTML, Layout
from IPython.display import display

# ============================================================
# FIR DESIGN BY THE WINDOW METHOD
# ============================================================

style_html = HTML("""
<style>
.fir2-root {width:970px; max-width:970px; font-family:Arial,sans-serif;}
.fir2-header {background:linear-gradient(90deg,#00695c,#26a69a); color:white; padding:10px 15px; border-radius:8px 8px 0 0; font-size:19px; font-weight:bold;}
.fir2-intro {background:#f2faf8; border:1px solid #b8dcd6; border-top:none; padding:8px 12px; border-radius:0 0 8px 8px; font-size:12px; line-height:1.5; margin-bottom:8px;}
.fir2-accent {font-weight:bold; color:#00695c;}
.fir2-title {font-size:12.5px; font-weight:bold; margin:0 0 5px 2px; color:#00695c;}
.jupyter-widgets-output-area, .widget-output, .output_area, .output_subarea {overflow-x:visible !important; max-width:none !important;}
</style>
""")

header_html = HTML("""
<div class="fir2-root">
<div class="fir2-header">FIR Design by the Window Method</div>
<div class="fir2-intro">
<span class="fir2-accent">Design process:</span>
select the desired frequency-selective response, choose the FIR length N and a window function, and observe the complete transformation from the ideal response to the realizable FIR filter.
<br>
<span class="fir2-accent">What to observe:</span>
windowing makes the ideal infinite impulse response finite, while the actual frequency response develops a transition band and ripples because multiplication in time corresponds to convolution in frequency.
</div>
</div>
""")

# ============================================================
# WINDOW FUNCTIONS
# ============================================================

def make_window(window_type, N):
    n = np.arange(N)
    if window_type == 'Rectangular': return np.ones(N)
    if window_type == 'Bartlett': return np.bartlett(N)
    if window_type == 'Hann': return 0.5 - 0.5 * np.cos(2 * np.pi * n / (N - 1))
    if window_type == 'Hamming': return 0.54 - 0.46 * np.cos(2 * np.pi * n / (N - 1))
    if window_type == 'Blackman': return 0.42 - 0.50 * np.cos(2 * np.pi * n / (N - 1)) + 0.08 * np.cos(4 * np.pi * n / (N - 1))

# ============================================================
# IDEAL LOW-PASS IMPULSE RESPONSE
# ============================================================

def ideal_lowpass(m, wc):
    hd = np.empty_like(m, dtype=float)
    center = np.abs(m) < 1e-12
    hd[center] = wc / np.pi
    hd[~center] = np.sin(wc * m[~center]) / (np.pi * m[~center])
    return hd

# ============================================================
# IDEAL IMPULSE RESPONSE
# ============================================================

def ideal_impulse_response(filter_type, N, f1, f2):
    n = np.arange(N)
    alpha = (N - 1) / 2.0
    m = n - alpha
    w1 = np.pi * f1
    w2 = np.pi * f2
    lp1 = ideal_lowpass(m, w1)

    if filter_type == 'Low-pass': return lp1

    if filter_type == 'High-pass':
        hd = -lp1.copy()
        center = np.abs(m) < 1e-12
        hd[center] += 1.0
        return hd

    lp2 = ideal_lowpass(m, w2)

    if filter_type == 'Band-pass': return lp2 - lp1

    hd = lp1 - lp2
    center = np.abs(m) < 1e-12
    hd[center] += 1.0
    return hd

# ============================================================
# IDEAL MAGNITUDE RESPONSE
# ============================================================

def ideal_frequency_response(filter_type, omega, f1, f2):
    w1 = np.pi * f1
    w2 = np.pi * f2
    Hd = np.zeros_like(omega)

    if filter_type == 'Low-pass': Hd[np.abs(omega) <= w1] = 1.0
    elif filter_type == 'High-pass': Hd[np.abs(omega) >= w1] = 1.0
    elif filter_type == 'Band-pass': Hd[(np.abs(omega) >= w1) & (np.abs(omega) <= w2)] = 1.0
    else: Hd[(np.abs(omega) <= w1) | (np.abs(omega) >= w2)] = 1.0

    return Hd

# ============================================================
# DTFT
# ============================================================

def calculate_dtft(h, omega):
    n = np.arange(len(h))
    return np.exp(-1j * np.outer(omega, n)) @ h

# ============================================================
# MAIN INTERACTIVE FUNCTION
# ============================================================

def plot_window_design(filter_type='Low-pass', window_type='Hann', N=41, f1=0.30, f2=0.60):
    if filter_type in ['Band-pass', 'Band-stop'] and f2 <= f1: f2 = min(f1 + 0.05, 0.95)

    n = np.arange(N)
    alpha = (N - 1) / 2.0
    hd = ideal_impulse_response(filter_type, N, f1, f2)
    w = make_window(window_type, N)
    h = hd * w

    omega = np.linspace(-np.pi, np.pi, 2400)
    Hd = ideal_frequency_response(filter_type, omega, f1, f2)
    H = calculate_dtft(h, omega)
    magnitude = np.abs(H)
    magnitude_db = 20 * np.log10(np.maximum(magnitude, 1e-6))

    fig = plt.figure(figsize=(14.8, 8.0))
    grid = fig.add_gridspec(2, 4, width_ratios=[1.0, 1.0, 1.35, 0.82], height_ratios=[1.0, 1.0], wspace=0.38, hspace=0.48)

    ax_hd = fig.add_subplot(grid[0, 0])
    ax_window = fig.add_subplot(grid[0, 1])
    ax_result = fig.add_subplot(grid[:, 2])
    ax_info = fig.add_subplot(grid[:, 3])
    ax_impulse = fig.add_subplot(grid[1, 0])
    ax_db = fig.add_subplot(grid[1, 1])

    # ========================================================
    # 1. DESIRED FREQUENCY RESPONSE
    # ========================================================

    ax_hd.plot(omega / np.pi, Hd, linewidth=1.8)
    ax_hd.set_xlim(-1, 1)
    ax_hd.set_ylim(-0.08, 1.15)
    ax_hd.set_xlabel(r'Normalized frequency $\omega/\pi$')
    ax_hd.set_ylabel(r'$H_d(e^{j\omega})$')
    ax_hd.set_title('1. Desired Frequency Response')
    ax_hd.grid(True, linestyle=':', alpha=0.25)

    # ========================================================
    # 2. WINDOW FUNCTION
    # ========================================================

    markerline, stemlines, baseline = ax_window.stem(n, w, basefmt=' ')
    plt.setp(stemlines, linewidth=1.1)
    plt.setp(markerline, markersize=3.8)

    ax_window.set_xlim(-1, N)
    ax_window.set_ylim(-0.08, 1.12)
    ax_window.set_xlabel('Sample index n')
    ax_window.set_ylabel(r'$w[n]$')
    ax_window.set_title(f'2. {window_type} Window')
    ax_window.grid(True, linestyle=':', alpha=0.25)

    # ========================================================
    # 3. IDEAL AND FINITE IMPULSE RESPONSES
    # ========================================================

    ax_impulse.plot(n, hd, 'o-', linewidth=1.0, markersize=3.4, label=r'Ideal shifted $h_d[n]$')
    ax_impulse.plot(n, h, 'o-', linewidth=1.4, markersize=3.4, label=r'Windowed $h[n]$')
    ax_impulse.axvline(alpha, linestyle=':', linewidth=1.0)

    ax_impulse.set_xlim(-1, N)
    ax_impulse.set_xlabel('Sample index n')
    ax_impulse.set_ylabel('Amplitude')
    ax_impulse.set_title(r'3. $h[n]=h_d[n]w[n]$')
    ax_impulse.grid(True, linestyle=':', alpha=0.25)
    ax_impulse.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=2, frameon=False, fontsize=9.2)

    # ========================================================
    # 4. ACTUAL RESPONSE IN dB
    # ========================================================

    ax_db.plot(omega / np.pi, magnitude_db, linewidth=1.5)
    ax_db.set_xlim(-1, 1)
    ax_db.set_ylim(-100, 5)
    ax_db.set_xlabel(r'Normalized frequency $\omega/\pi$')
    ax_db.set_ylabel('Magnitude [dB]')
    ax_db.set_title('4. Actual Response in dB')
    ax_db.grid(True, linestyle=':', alpha=0.25)

    # ========================================================
    # 5. IDEAL VERSUS ACTUAL RESPONSE
    # ========================================================

    ax_result.plot(omega / np.pi, Hd, '--', linewidth=1.7, label='Ideal response')
    ax_result.plot(omega / np.pi, magnitude, linewidth=1.8, label='Actual FIR response')

    ax_result.set_xlim(-1, 1)
    ax_result.set_ylim(-0.10, max(1.20, 1.08 * np.max(magnitude)))
    ax_result.set_xlabel(r'Normalized frequency $\omega/\pi$', labelpad=9)
    ax_result.set_ylabel('Magnitude')
    ax_result.set_title('5. Ideal vs Actual FIR Response', fontsize=12)
    ax_result.grid(True, linestyle=':', alpha=0.25)
    ax_result.legend(loc='upper center', bbox_to_anchor=(0.5, -0.10), ncol=2, frameon=False, fontsize=9.8)

    # ========================================================
    # DESIGN INFORMATION PANEL
    # ========================================================

    ax_info.axis('off')

    if filter_type in ['Low-pass', 'High-pass']:
        frequency_text = f'Cutoff frequency\n{f1:.2f}π'
    else:
        frequency_text = f'Cutoff frequencies\n{f1:.2f}π, {f2:.2f}π'

    info_text = (
        f'DESIGN INFORMATION\n'
        f'────────────────────\n'
        f'Filter type\n{filter_type}\n\n'
        f'Window\n{window_type}\n\n'
        f'Length N\n{N}\n\n'
        f'Filter order\n{N - 1}\n\n'
        f'Delay α\n{alpha:g}\n\n'
        f'{frequency_text}\n\n'
        f'WINDOW METHOD\n'
        f'────────────────────\n'
        f'h[n] = hd[n] w[n]\n'
        f'α = (N-1)/2'
    )

    ax_info.text(0.00, 1.00, info_text, transform=ax_info.transAxes, ha='left', va='top', fontsize=10.4, family='monospace', linespacing=1.32)

    fig.suptitle('FIR Design by the Window Method', fontsize=13.5)
    plt.subplots_adjust(left=0.045, right=0.985, top=0.91, bottom=0.13)
    plt.show()
    plt.close(fig)

# ============================================================
# CONTROLS
# ============================================================

filter_selector = Dropdown(options=['Low-pass', 'High-pass', 'Band-pass', 'Band-stop'], value='Low-pass', description='Filter:', style={'description_width':'55px'}, layout=Layout(width='235px'))
window_selector = Dropdown(options=['Rectangular', 'Bartlett', 'Hann', 'Hamming', 'Blackman'], value='Hann', description='Window:', style={'description_width':'60px'}, layout=Layout(width='235px'))

N_slider = IntSlider(value=41, min=11, max=101, step=2, description='Length N:', continuous_update=True, style={'description_width':'68px'}, layout=Layout(width='285px'))
f1_slider = FloatSlider(value=0.30, min=0.05, max=0.90, step=0.01, description='Frequency 1:', continuous_update=True, readout_format='.2f', style={'description_width':'82px'}, layout=Layout(width='300px'))
f2_slider = FloatSlider(value=0.60, min=0.10, max=0.95, step=0.01, description='Frequency 2:', continuous_update=True, readout_format='.2f', style={'description_width':'82px'}, layout=Layout(width='300px'))

# ============================================================
# ENABLE / DISABLE SECOND FREQUENCY
# ============================================================

def update_frequency_controls(change=None):
    f2_slider.disabled = filter_selector.value in ['Low-pass', 'High-pass']

filter_selector.observe(update_frequency_controls, names='value')
update_frequency_controls()

# ============================================================
# INTERACTIVE OBJECT
# ============================================================

widget_plot = interactive(plot_window_design, filter_type=filter_selector, window_type=window_selector, N=N_slider, f1=f1_slider, f2=f2_slider)
plot_output = widget_plot.children[-1]
plot_output.layout = Layout(width='auto', overflow='visible')

# ============================================================
# CONTROL LAYOUT
# ============================================================

selection_box = VBox([HTML("<div class='fir2-title'>Filter and Window Selection</div>"), HBox([filter_selector, window_selector], layout=Layout(width='490px', justify_content='space-between'))], layout=Layout(width='970px', border='1px solid #b8dcd6', padding='7px 10px', overflow='visible'))

parameter_row = HBox([N_slider, f1_slider, f2_slider], layout=Layout(width='930px', justify_content='space-between', align_items='center'))
parameter_box = VBox([HTML("<div class='fir2-title'>Design Parameters</div>"), parameter_row], layout=Layout(width='970px', border='1px solid #b8dcd6', padding='7px 10px', overflow='visible'))

main_layout = VBox([header_html, selection_box, parameter_box, plot_output], layout=Layout(width='970px', overflow='visible', align_items='flex-start'))

display(style_html)
display(main_layout)